# **Beachcast degradation detection using PlanetScope and Sentinel-2 imagery**

*This code has been developed for detecting differences in Zostera noltii and Rugulopteryx okamurae beachcasts in Cadiz province.*

> **Mar Roca Mora**
Last update: 2025/12/01



## Libraries

In [ ]:
!pip install xlsxwriter
!pip install geemap
!pip install netCDF4
!pip install rioxarray
!pip install earthengine-api
!pip install scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.8 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


# 0 - Set up

In [ ]:
## Connection to GEE project
import ee
ee.Authenticate()
ee.Initialize(project='ee-seagrass-mar')

## Connection to Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
## Import some libraries
import os
import ee
import geemap
import geemap.colormaps as cm
import netCDF4
import pandas as pd
import xarray as xr
import numpy as np
import rioxarray
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
geemap.ee_initialize()

In [ ]:
## Clone GitHub repo:
##!git clone https://github.com/marroca13/seagrass_PS_S2.git

# 1 - BOA imagery

In [ ]:
## PlanetScope data
PS_Zostera = ee.Image('projects/ee-seagrass-mar/assets/Planet_250612_Zostera').divide(10000) # 8-bands 2259x4305 px
bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8']
PS_Zostera = PS_Zostera.select(PS_Zostera.bandNames()).rename(bands)

PS_Rug = ee.Image('projects/ee-seagrass-mar/assets/Planet_250613_Rugulopteryx').divide(10000) # 8-bands 2259x4305 px
bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8']
PS_Rug = PS_Rug.select(PS_Rug.bandNames()).rename(bands)

vis_params_PS = {'bands': ['B6', 'B4', 'B2'], 'gamma': 1.4, 'min': 0.06067177176193392, 'max': 0.17756883760596057}

In [ ]:
## Sentinel-2 data
S2_11_id = ee.Image('COPERNICUS/S2_SR_HARMONIZED/20250611T110711_20250611T111248_T30STF').divide(10000).select('B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12')
S2_14_id = ee.Image('COPERNICUS/S2_SR_HARMONIZED/20250614T110619_20250614T111542_T30STF').divide(10000).select('B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12')

## Resample all bands to 10 m
S2_11_id = S2_11_id.reproject(crs='EPSG:4326', scale=10).select('B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12')
S2_14_id = S2_14_id.reproject(crs='EPSG:4326', scale=10).select('B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12')

# Display imagery in map
vis_params_S2 = {'bands': ['B4', 'B3', 'B2'], 'gamma': 1.4, 'min': 0.01647413318891601, 'max': 0.24818905748819892}

In [ ]:
# # Step 1: Get the projection (CRS) of the PS_Zostera image
# zostera_projection = PS_Zostera.projection()

# # Step 2: Reproject the Sentinel-2 image (S2_11_id) to match the projection of PS_Zostera
# S2_11_id = S2_11_id.reproject(
#     crs=zostera_projection,  # Reproject to match PS_Zostera's CRS
#     scale=10  # Use a resolution of 10m (if it matches your desired output)
# )

# # Step 2: Reproject the Sentinel-2 image (S2_11_id) to match the projection of PS_Zostera
# S2_14_id = S2_14_id.reproject(
#     crs=zostera_projection,  # Reproject to match PS_Zostera's CRS
#     scale=10  # Use a resolution of 10m (if it matches your desired output)
# )

In [ ]:
# Points for transects
Points_Zostera = ee.FeatureCollection('projects/ee-seagrass-mar/assets/Points_Zostera')
Points_Rug = ee.FeatureCollection('projects/ee-seagrass-mar/assets/Point_Rugulopteryx')
Point_Zostera_S2 = ee.FeatureCollection('projects/ee-seagrass-mar/assets/Point_Zostera_S2')
Point_Rug_S2 = ee.FeatureCollection('projects/ee-seagrass-mar/assets/Point_Rug_S2')
Points_Zostera

### 1.1 - *Zostera noltii*

In [ ]:
Map = geemap.Map()
Map.addLayer(PS_Zostera, vis_params_PS, "PS Zostera")
Map.addLayer(S2_11_id, vis_params_S2, "Sentinel-2 Zostera")
Map.addLayer(Points_Zostera, {'color': '#87b36f'}, 'Points_Zostera')
Map.addLayer(Point_Zostera_S2, {'color': 'blue'}, 'Point_Zostera_S2')
Map.centerObject(Points_Zostera, zoom=19)
Map

NameError: name 'geemap' is not defined

In [ ]:
# Add latitude and longitude properties to each point feature
Points_Zostera_withXY = Points_Zostera.map(
    lambda f: f.set({
        'latitude':  f.geometry().coordinates().get(1),
        'longitude': f.geometry().coordinates().get(0)
    })
)

PS_Zostera_values = PS_Zostera.sampleRegions(
    collection=Points_Zostera_withXY,
    properties=['system:index', 'latitude', 'longitude'],
    scale=3
)

PS_Zostera_values

import ee
task = ee.batch.Export.table.toDrive(
    collection=PS_Zostera_values,
    folder = 'Okamurae',
    description='PS_Zostera_values_coords',
    fileFormat='CSV'
)

Points_Zostera_withXY

# task.start()


In [ ]:
# Extract pixel values from PlanetScope imagery for Rugulopteryx points
Points_Zostera_withXY_S2 = Point_Zostera_S2.map(
    lambda f: f.set({
        'latitude':  f.geometry().coordinates().get(1),
        'longitude': f.geometry().coordinates().get(0)
    })
)

S2_Zostera_values = S2_11_id.sampleRegions(
    collection=Points_Zostera_withXY_S2,
    properties=['system:index', 'latitude', 'longitude'],
    scale=10
)

S2_Zostera_values

import ee

task = ee.batch.Export.table.toDrive(
    collection=S2_Zostera_values,
    folder = 'Okamurae',
    description='S2_Zostera_values_B12',
    fileFormat='CSV'
)

S2_Zostera_values

#task.start()

In [ ]:
## Download data
s2_zostera_list = S2_Zostera_values.getInfo()['features']

extracted_data = []
for feature in s2_zostera_list:
    extracted_data.append(feature['properties'])

df_s2_zostera = pd.DataFrame(extracted_data, columns=bands)
df_s2_zostera.head()

from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_s2_zostera)

https://docs.google.com/spreadsheets/d/1K7YeBh6fA7Dw-63c7-beFBLEw9jUB7zuDQJNwCaM2xg/edit#gid=0


### 1.2 - *Rugulopteryx okamurae*

In [ ]:
Map = geemap.Map()
Map.addLayer(PS_Rug, vis_params_PS, "PS Rug")
Map.addLayer(S2_14_id, vis_params_S2, "Sentinel-2 Rug")
Map.addLayer(Points_Rug, {'color': 'orange'}, 'Points_Rug')
Map.addLayer(Point_Rug_S2, {'color': 'blue'}, 'Point_Rug_S2')
Map.centerObject(Points_Rug, zoom=19)
Map

Map(center=[36.07080463251376, -5.748779787233997], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Add latitude and longitude properties to each point feature
Points_Rug_withXY = Points_Rug.map(
    lambda f: f.set({
        'latitude':  f.geometry().coordinates().get(1),
        'longitude': f.geometry().coordinates().get(0)
    })
)

PS_Rug_values = PS_Rug.sampleRegions(
    collection=Points_Rug_withXY,
    properties=['system:index', 'latitude', 'longitude'],
    scale=3
)

PS_Rug_values

import ee
task = ee.batch.Export.table.toDrive(
    collection=PS_Rug_values,
    folder = 'Okamurae',
    description='PS_Rug_values',
    fileFormat='CSV'
)

task.start()

In [ ]:
# # Extract pixel values from PlanetScope imagery for Rugulopteryx points
# Points_Rug_withXY_S2 = Point_Rug_S2.map(
#     lambda f: f.set({
#         'latitude':  f.geometry().coordinates().get(1),
#         'longitude': f.geometry().coordinates().get(0)
#     })
# )

# S2_Rug_values = S2_14_id.sampleRegions(
#     collection=Points_Rug_withXY_S2,
#     properties=['system:index', 'latitude', 'longitude'],
#     scale=10
# )

# S2_Rug_values

# import ee
# task = ee.batch.Export.table.toDrive(
#     collection=S2_Rug_values,
#     folder = 'Okamurae',
#     description='S2_Rug_values_B12',
#     fileFormat='CSV'
# )

# task.start()

In [ ]:
s2_zostera_list = S2_Zostera_values.getInfo()['features']

extracted_s2_data = []
for feature in s2_zostera_list:
    extracted_s2_data.append(feature['properties'])

# Create a list of expected band names for Sentinel-2
s2_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12']

# # Filter out properties that are not band names or system:index
# # Also handle cases where some bands might be missing (e.g., B1, B9, B10 are 60m bands, B8A is 20m)
# # For simplicity, we'll try to include all potential bands and let NaNs appear if a band isn't present in the image.

# df_s2_zostera = pd.DataFrame(extracted_s2_data)

# # Display the DataFrame
# display(df_s2_zostera.head())

## NDVI and NDMI indeces

In [ ]:
## Calculate NDVI

def calculate_ndvi(image, nir_band, red_band):
    """
    Calculates NDVI (Normalized Difference Vegetation Index).
    """
    ndvi = image.normalizedDifference([nir_band, red_band]).rename('NDVI')
    return ndvi

# Calculate NDVI for PlanetScope Zostera
# Using B8 (NIR) and B6 (Red) for PlanetScope SuperDove
PS_Zostera_ndvi = calculate_ndvi(PS_Zostera, 'B8', 'B6')

# Calculate NDVI for PlanetScope Rugulopteryx
# Using B8 (NIR) and B6 (Red) for PlanetScope SuperDove
PS_Rug_ndvi = calculate_ndvi(PS_Rug, 'B8', 'B6')

# Calculate NDVI for Sentinel-2 (June 11)
# Using B8 (NIR) and B4 (Red) for Sentinel-2
S2_11_ndvi = calculate_ndvi(S2_11_id, 'B8', 'B4')

# Calculate NDVI for Sentinel-2 (June 14)
# Using B8 (NIR) and B4 (Red) for Sentinel-2
S2_14_ndvi = calculate_ndvi(S2_14_id, 'B8', 'B4')

print("NDVI calculation complete for all images.")


NDVI calculation complete for all images.


In [ ]:
Map_ndvi = geemap.Map()

ndvi_vis = {
    'min': -0.1,
    'max': 0.8,
    'palette':[
  '#FFFFFF',
  '#CEEAB1',
  '#8FCE00',
  '#228B22',
  '#006400']}

Map_ndvi.addLayer(PS_Zostera_ndvi, ndvi_vis, 'PS_Zostera_NDVI')
Map_ndvi.addLayer(PS_Rug_ndvi, ndvi_vis, 'PS_Rug_NDVI')
Map_ndvi.addLayer(S2_11_ndvi, ndvi_vis, 'S2_11_NDVI')
Map_ndvi.addLayer(S2_14_ndvi, ndvi_vis, 'S2_14_NDVI')
Map_ndvi.addLayer(Points_Rug, {'color': 'orange'}, 'Points_Rug')
Map_ndvi.addLayer(Point_Rug_S2, {'color': 'blue'}, 'Point_Rug_S2')
Map_ndvi.addLayer(Points_Zostera, {'color': '#87b36f'}, 'Points_Zostera')
Map_ndvi.addLayer(Point_Zostera_S2, {'color': 'blue'}, 'Point_Zostera_S2')

# Center the map on one of the points for better viewing, e.g., Points_Zostera
Map_ndvi.centerObject(Points_Rug, zoom=12)
Map_ndvi

Map(center=[36.07080463251376, -5.748779787233997], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
## Calcualate NDMI for S2
def calculate_ndmi(image, nir_band, swir_band):
    """
    Calculates NDMI (Normalized Difference Moisture Index).
    """
    ndmi = image.normalizedDifference([nir_band, swir_band]).rename('NDMI')
    return ndmi

# Calculate NDMI for Sentinel-2 (June 11) using B8 (NIR) and B11 (SWIR)
S2_11_ndmi = calculate_ndmi(S2_11_id, 'B8', 'B11')

# Calculate NDMI for Sentinel-2 (June 14) using B8 (NIR) and B11 (SWIR)
S2_14_ndmi = calculate_ndmi(S2_14_id, 'B8', 'B11')

print("NDMI calculation complete for Sentinel-2 images.")

# Add NDVI and NDMI as new bands to the original Sentinel-2 images
S2_11_id = S2_11_id.addBands(S2_11_ndvi).addBands(S2_11_ndmi)
S2_14_id = S2_14_id.addBands(S2_14_ndvi).addBands(S2_14_ndmi)

print("NDVI and NDMI bands added to Sentinel-2 imagery.")

NDMI calculation complete for Sentinel-2 images.
NDVI and NDMI bands added to Sentinel-2 imagery.


In [ ]:
## map NDMI values
Map_ndmi = geemap.Map()

# Reusing ndvi_vis for NDMI visualization as the range is similar (-1 to 1)
ndmi_vis = {
    'min': -0.3,
    'max': 0.6,
    'palette':[
  '#FFFFFF',
  '#CCEEFF',
  '#77BBFF',
  '#0044FF',
  '#000088']}

Map_ndmi.addLayer(S2_11_ndmi, ndmi_vis, 'S2_11_NDMI')
Map_ndmi.addLayer(S2_14_ndmi, ndmi_vis, 'S2_14_NDMI')
Map_ndmi.addLayer(Points_Rug, {'color': 'orange'}, 'Points_Rug')
Map_ndmi.addLayer(Points_Zostera, {'color': '#87b36f'}, 'Points_Zostera')

# Center the map on one of the points for better viewing, e.g., Points_Zostera
Map_ndmi.centerObject(Points_Rug, zoom=17)
Map_ndmi

Map(center=[36.07080463251376, -5.748779787233997], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
Points_Zostera_withXY

In [ ]:
import pandas as pd

# Extract NDVI values from PlanetScope Zostera for Zostera points
PS_Zostera_ndvi_values = PS_Zostera_ndvi.sampleRegions(
    collection=Points_Zostera_withXY,
    properties=['system:index', 'latitude', 'longitude'],
    scale=3
)

# Convert to pandas DataFrame
ps_zostera_ndvi_list = PS_Zostera_ndvi_values.getInfo()['features']
extracted_ps_zostera_ndvi = []
for feature in ps_zostera_ndvi_list:
    extracted_ps_zostera_ndvi.append(feature['properties'])
df_ps_zostera_ndvi = pd.DataFrame(extracted_ps_zostera_ndvi)

# Display the full DataFrame to show all 10 values
display(df_ps_zostera_ndvi)

,NDVI,latitude,longitude
0,0.293046,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"
1,0.261821,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"
2,0.247690,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"
3,0.269841,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"
4,0.324878,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"
5,0.394799,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"
6,0.405578,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"
7,0.420732,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"
8,0.392333,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"
9,0.434551,"[-6.211548069283618, 36.47604253661351]","[-6.2116084189864536, 36.47598646033817]"


In [ ]:
# Extract NDVI values from PlanetScope Rugulopteryx for Rugulopteryx points
PS_Rug_ndvi_values = PS_Rug_ndvi.sampleRegions(
    collection=Points_Rug_withXY,
    properties=['system:index', 'latitude', 'longitude'],
    scale=3
)

# Convert to pandas DataFrame
ps_rug_ndvi_list = PS_Rug_ndvi_values.getInfo()['features']
extracted_ps_rug_ndvi = []
for feature in ps_rug_ndvi_list:
    extracted_ps_rug_ndvi.append(feature['properties'])
df_ps_rug_ndvi = pd.DataFrame(extracted_ps_rug_ndvi)
display(df_ps_rug_ndvi)

,NDVI,latitude,longitude
0,0.458982,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"
1,0.449028,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"
2,0.456922,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"
3,0.423779,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"
4,0.295417,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"
5,0.332012,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"
6,0.354258,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"
7,0.388875,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"
8,0.405682,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"
9,0.426901,"[-5.748785676538595, 36.070887679618245]","[-5.748779349837272, 36.07077427002355]"


In [ ]:
# Extract all bands (including NDVI and NDMI) from S2_11_id for Zostera points
S2_Zostera_indices_values = S2_11_id.sampleRegions(
    collection=Point_Zostera_S2,
    properties=['system:index', 'latitude', 'longitude'],
    scale=10
)

# Convert to pandas DataFrame
s2_zostera_indices_list = S2_Zostera_indices_values.getInfo()['features']
extracted_s2_zostera_indices = []
for feature in s2_zostera_indices_list:
    extracted_s2_zostera_indices.append(feature['properties'])
df_s2_zostera_indices = pd.DataFrame(extracted_s2_zostera_indices)
display(df_s2_zostera_indices)

,B1,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,B9,NDMI,NDVI
0,0.0494,0.3719,0.2748,0.1160,0.1600,0.2152,0.2238,0.2770,0.3116,0.3468,0.3647,0.2665,-0.034924,0.234164
1,0.0426,0.2366,0.1666,0.0191,0.0466,0.0600,0.1181,0.1558,0.1831,0.1828,0.2173,0.2180,-0.128278,0.505766
2,0.0426,0.2366,0.1666,0.0187,0.0444,0.0553,0.1181,0.1558,0.1831,0.1724,0.2173,0.2180,-0.156968,0.514273
3,0.0494,0.2779,0.1983,0.0530,0.0867,0.1214,0.1507,0.1853,0.2054,0.2420,0.2426,0.2665,-0.069052,0.331866


In [ ]:
# Extract all bands (including NDVI and NDMI) from S2_14_id for Rugulopteryx points
S2_Rug_indices_values = S2_14_id.sampleRegions(
    collection=Point_Rug_S2,
    properties=['system:index', 'latitude', 'longitude'],
    scale=10
)

# Convert to pandas DataFrame
s2_rug_indices_list = S2_Rug_indices_values.getInfo()['features']
extracted_s2_rug_indices = []
for feature in s2_rug_indices_list:
    extracted_s2_rug_indices.append(feature['properties'])
df_s2_rug_indices = pd.DataFrame(extracted_s2_rug_indices)
display(df_s2_rug_indices)

,B1,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,B9,NDMI,NDVI
0,0.0571,0.1201,0.0853,0.0465,0.0624,0.0702,0.1147,0.1209,0.1361,0.1194,0.1378,0.2258,-0.002923,0.259494
1,0.0571,0.2843,0.2062,0.0855,0.1514,0.1600,0.2109,0.2654,0.2775,0.2708,0.2938,0.2258,-0.024320,0.257196
2,0.0571,0.2232,0.1629,0.0512,0.0836,0.1142,0.1910,0.2018,0.2201,0.2144,0.2446,0.2258,-0.020110,0.304930
3,0.0571,0.1201,0.0853,0.0349,0.0563,0.0706,0.1147,0.1209,0.1361,0.1442,0.1378,0.2258,0.091184,0.342644
